# Dataset Exploration and Partioning
  

In [1]:
# ===============================================
# BASIC SETUP: GIT Auth and mounting google drive
# ================================================

from google.colab import drive
import sys

# Standard mount to access the config file initially
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/Deep-Learning-Final-Project/CNN-vs-VIT-Agribench/src')

from utils.config import initialize_project
initialize_project()

Mounted at /content/drive
Authenticated as: Ande404
Working Directory: /content/drive/MyDrive/Deep-Learning-Final-Project/CNN-vs-VIT-Agribench


In [2]:
# ===============================================
# Data Exploration: how many classes do we have
# ================================================
import os
from collections import Counter

data_dir = "data/raw/PlantVillage"

classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
print("Number of classes:", len(classes))
print("Sample classes:", classes[:10])

class_counts = {}
for cls in classes:
    cls_path = os.path.join(data_dir, cls)
    class_counts[cls] = len([
        f for f in os.listdir(cls_path)
        if os.path.isfile(os.path.join(cls_path, f))
    ])

print("Min class size:", min(class_counts.values()))
print("Max class size:", max(class_counts.values()))

Number of classes: 15
Sample classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot']
Min class size: 152
Max class size: 3209


We are only going to use these 5 classes:


*   Pepper__bell___Bacterial_spot
*   Pepper__bell___healthy
*   Potato___Early_blight
*   Potato___Late_blight  
*   Potato___healthy  



In [3]:
# =====================================================================
# Data Subset Creation: Extract those classes and make a subset subdir
# =====================================================================

import os
import shutil

raw_dir = "data/raw/PlantVillage"
subset_dir = "data/subset/PlantVillage"

selected_classes = [
    "Pepper__bell___Bacterial_spot",
    "Pepper__bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy"
]

os.makedirs(subset_dir, exist_ok=True)

for cls in selected_classes:
    src = os.path.join(raw_dir, cls)
    dst = os.path.join(subset_dir, cls)

    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Copied: {cls}")
    else:
        print(f"Missing: {cls}")

Copied: Pepper__bell___Bacterial_spot
Copied: Pepper__bell___healthy
Copied: Potato___Early_blight
Copied: Potato___Late_blight
Copied: Potato___healthy


In [6]:
## verify the classes exist and the counts of images in each class
subset_classes = [d for d in os.listdir(subset_dir) if os.path.isdir(os.path.join(subset_dir, d))]
print("Subset classes:", subset_classes)
print("Number of subset classes:", len(subset_classes))

subset_counts = {}

for cls in subset_classes:
    cls_path = os.path.join(subset_dir, cls)
    subset_counts[cls] = len([
        f for f in os.listdir(cls_path)
        if os.path.isfile(os.path.join(cls_path, f))
    ])

print(subset_counts)

Subset classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']
Number of subset classes: 5
{'Pepper__bell___Bacterial_spot': 997, 'Pepper__bell___healthy': 1478, 'Potato___Early_blight': 1000, 'Potato___Late_blight': 1000, 'Potato___healthy': 152}


## Data Spliting
  - We shall split data into 80% Training, 10% Validation and 10% Split
  - We shall use a stratified split
  

In [7]:
# =================
# Data Spliting
# =================
import random

random.seed(42)

subset_dir = "data/subset/PlantVillage"
processed_dir = "data/processed/PlantVillage"

splits = ["train", "val", "test"]

# create split folders
for split in splits:
    os.makedirs(os.path.join(processed_dir, split), exist_ok=True)

for class_name in os.listdir(subset_dir):
    class_path = os.path.join(subset_dir, class_name)
    images = os.listdir(class_path)

    random.shuffle(images)

    n = len(images)
    train_end = int(0.8 * n)
    val_end = int(0.9 * n)

    split_data = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    for split, split_images in split_data.items():
        split_class_dir = os.path.join(processed_dir, split, class_name)
        os.makedirs(split_class_dir, exist_ok=True)

        for img in split_images:
            src = os.path.join(class_path, img)
            dst = os.path.join(split_class_dir, img)
            shutil.copy(src, dst)

print("Data split complete!")

Data split complete!


In [8]:
def count_images(base_dir):
    for split in ["train", "val", "test"]:
        print(f"\n--- {split.upper()} ---")
        split_path = os.path.join(base_dir, split)

        for cls in os.listdir(split_path):
            cls_path = os.path.join(split_path, cls)
            count = len(os.listdir(cls_path))
            print(f"{cls}: {count}")

count_images("data/processed/PlantVillage")


--- TRAIN ---
Pepper__bell___Bacterial_spot: 797
Pepper__bell___healthy: 1182
Potato___Early_blight: 800
Potato___Late_blight: 800
Potato___healthy: 121

--- VAL ---
Pepper__bell___Bacterial_spot: 100
Pepper__bell___healthy: 148
Potato___Early_blight: 100
Potato___Late_blight: 100
Potato___healthy: 15

--- TEST ---
Pepper__bell___Bacterial_spot: 100
Pepper__bell___healthy: 148
Potato___Early_blight: 100
Potato___Late_blight: 100
Potato___healthy: 16
